In [ ]:
import os
import random
import time
from pathlib import Path
import subprocess
from IPython.display import Video, display

import pandas as pd
import numpy as np
import torch
from decord import VideoReader
from PIL import Image
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

VIDEO_DIR = "/work/cs-503/sadgal/Videos_mp4"
GT_DIR = "/work/cs-503/sadgal/Annotation"
OUTPUT_DIR = "outputs"

DATASET_FPS = 25
CLIP_DURATION_SEC = 60
CLIP_DURATION_FRAMES = CLIP_DURATION_SEC * DATASET_FPS
NFRAMES_PER_CLIP = 35

VIDEO_IDS = [
    "P10T11C03",
    "P11T03C04",
    "P14T05C05",
    "P16T14C06",
    "P18T11C02"
]

_CLIP_WINDOWS: dict[str, tuple[int, int]] = {
    "P10T11C03": (22875, 24375),
    "P11T03C04": (18250, 19750),
    "P14T05C05": (9000, 10500),
    "P16T14C06": (12125, 13625),
    "P18T11C02": (19125, 20625),
}

CLASSES_TO_DROP = {"Breakfast", "Cook", "Make_coffee", "Make_tea"}
CLASSES_MAPPING = {
    "Breakfast.Cut_bread" : "cut bread", "Breakfast.Eat_at_table" : "eat his/her breakfast", 
    "Breakfast.Spread_jam_or_butter" : "spread jam or butter on bread", "Breakfast.Take_ham" : "take ham", 
    "Clean_dishes" : "clean dishes", "Clean_dishes.Clean_with_water" : "rince dishes with water", 
    "Clean_dishes.Dry_up" : "dry up dishes", "Clean_dishes.Put_something_in_sink" : "put something in the sink", 
    "Cook.Cut" : "cut food", "Cook.Stir" : "mix something", "Cook.Use_oven" : "use oven", 
    "Cook.Use_stove" : "use stove", "Drink.From_bottle" : "drink from a bottle", "Drink.From_can" : "drink from a can", 
    "Drink.From_cup" : "drink from a cup", "Drink.From_glass" : "drink from a glass", 
    "Dump_in_trash" : "dump waste in the trash", "Eat_snack" : "eat a snack", "Enter" : "enter the room", 
    "Get_up" : "get up", "Get_water" : "get water from the tap", "Insert_tea_bag" : "insert a tea bag in a cup", 
    "Lay_down" : "lay down", "Leave" : "leave the room", "Make_coffee.Pour_grains" : "pour coffee grains", 
    "Make_coffee.Pour_water" : "pour water to make coffee", "Make_tea.Boil_water" : "boil water", 
    "Pour.From_bottle" : "pour liquid from bottle", "Pour.From_can" : "pour liquid from can", 
    "Pour.From_kettle" : "pour water from kettle", "Put_something_on_table" : "put something on a table", 
    "Read" : "read", "Sit_down" : "sit down", "Stir_coffee/tea" : "stir coffee or tea", "Take_pills" : "take pills", 
    "Take_something_off_table" : "take something off a table", "Use_Drawer" : "use a drawer (until it is closed)", 
    "Use_cupboard" : "use a cupboard (until it is closed)", "Use_fridge" : "use the fridge (until it is closed)", 
    "Use_glasses" : "put on or take off glasses", "Use_laptop" : "use a laptop", "Use_tablet" : "use a tablet", 
    "Use_telephone" : "use a telephone", "Walk" : "walk", "Watch_TV" : "watch the TV", "Wipe_table" : "wipe a surface", 
    "Write" : "hold a pen"
}

MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
os.environ["HF_HOME"] = "/work/cs-503/sadgal/hf_models_cache"

In [ ]:
def get_annotation_path(video_id: str) -> Path:
    # Extract subject id e.g. P14 from P14T05C05
    subject_id = video_id[:3]
    return Path(f"{GT_DIR}/{subject_id}/{video_id}.csv")



def load_annotations(video_id: str) -> pd.DataFrame:
    path = get_annotation_path(video_id)
    df = pd.read_csv(path)
    df["duration_frames"] = df["end_frame"] - df["start_frame"]
    df["duration_sec"] = df["duration_frames"] / DATASET_FPS

    # Drop parent composite classes
    df = df[~df["event"].isin(CLASSES_TO_DROP)].copy()

    # Drop classes not in mapping (unknown or unannotated)
    df = df[df["event"].isin(CLASSES_MAPPING)].copy()

    # Add human-readable description
    df["event_description"] = df["event"].map(CLASSES_MAPPING)

    return df.reset_index(drop=True)
    

def find_densest_clip(df: pd.DataFrame, 
                      clip_duration_frames: int = CLIP_DURATION_FRAMES,
                      step_frames: int = 25) -> dict:
    """
    Slide a window of clip_duration_frames across the video.
    Score each position by number of distinct event classes present.
    Returns the best window as a dict with start/end frames and events inside.
    """
    if df.empty:
        return None

    video_end = df["end_frame"].max()
    best_score = -1
    best_start = 0

    for start in range(0, int(video_end - clip_duration_frames), step_frames):
        end = start + clip_duration_frames

        # Events that overlap this window
        overlapping = df[
            (df["start_frame"] < end) & (df["end_frame"] > start)
        ]
        # Score: number of distinct classes
        score = overlapping["event"].nunique()

        if score > best_score:
            best_score = score
            best_start = start

    best_end = best_start + clip_duration_frames
    clip_events = df[
        (df["start_frame"] < best_end) & (df["end_frame"] > best_start)
    ].copy()

    return {
        "video_id":    None,   # filled below
        "start_frame": best_start,
        "end_frame":   best_end,
        "start_sec":   best_start / DATASET_FPS,
        "end_sec":     best_end   / DATASET_FPS,
        "n_classes":   best_score,
        "events":      clip_events.to_dict("records"),
    }

# Run for all videos
clips = {}
for vid_id in VIDEO_IDS:
    df = load_annotations(vid_id)
    clip = find_densest_clip(df)
    clip["video_id"] = vid_id
    clips[vid_id] = clip
    print(f"{vid_id}: clip {clip['start_sec']:.0f}s–{clip['end_sec']:.0f}s "
          f"| {clip['n_classes']} distinct classes "
          f"| {len(clip['events'])} event instances")

In [ ]:
def get_video_path(video_id: str) -> str:
    return f"{VIDEO_DIR}/{video_id}.mp4"

def extract_clip_frames(video_id: str, 
                        start_frame: int, 
                        end_frame: int, 
                        nframes: int = NFRAMES_PER_CLIP,
                        seed: int = 42) -> dict:
    """
    Returns frames for all three conditions:
      - normal:   chronological uniform sample
      - shuffled: same frames, random order
      - blind:    empty list
    """
    vr = VideoReader(get_video_path(video_id))
    indices = np.linspace(start_frame, end_frame - 1, nframes, dtype=int)
    frames_np = vr.get_batch(indices).asnumpy()
    pil_frames = [Image.fromarray(f) for f in frames_np]

    shuffled = pil_frames.copy()
    random.seed(seed)
    random.shuffle(shuffled)

    return {
        "normal":   pil_frames,
        "shuffled": shuffled,
        "blind":    [],
        "indices":  indices,   # keep for frame map in prompt
    }

# Extract for all clips
clip_frames = {}
for vid_id, clip in clips.items():
    clip_frames[vid_id] = extract_clip_frames(
        vid_id, clip["start_frame"], clip["end_frame"]
    )
    print(f"{vid_id}: {len(clip_frames[vid_id]['normal'])} frames extracted "
          f"(frames {clip['start_frame']}–{clip['end_frame']})")

In [ ]:
def sec_to_mmss(seconds: float) -> str:
    mm, ss = int(seconds // 60), seconds % 60
    return f"{mm:02d}:{ss:05.2f}"

def generate_questions(clip: dict, n_per_family: int = 5) -> list[dict]:
    """
    Generate 4 families of questions (ordering, counting, duration comparison, and
    composite activity identification) from GT annotations.
    """
    events = clip["events"]
    fps    = DATASET_FPS
    questions = []

    # ── Family 1: Ordering ────────────────────────────────────────────────────
    # Find non-overlapping pairs
    non_overlapping = [
        (a, b)
        for i, a in enumerate(events)
        for b in events[i+1:]
        if a["end_frame"] <= b["start_frame"] 
        and a["event"] != b["event"]
    ]
    random.shuffle(non_overlapping)
    for a, b in non_overlapping[:n_per_family]:
        questions.append({
            "family":   "ordering",
            "question": f"Does the person {b['event_description']} or {a['event_description']} first ?",
            "answer":   a['event_description'],
            "event_a":  a['event_description'],
            "event_b":  a['event_description'],
        })

    # ── Family 2: Counting ────────────────────────────────────────────────────
    from collections import Counter
    cls_desc_counts = Counter(e["event_description"] for e in events)
    # Only ask about classes that appear more than once
    multi_classes = [(cls_desc, cnt) for cls_desc, cnt in cls_desc_counts.items() if cnt > 1]
    random.shuffle(multi_classes)
    for cls_desc, cnt in multi_classes[:n_per_family]:
        questions.append({
            "family":   "counting",
            "question": f"How many times does the person {cls_desc} in the video?",
            "answer":   cnt,
            "event":    cls_desc,
        })

    # ── Family 3: Duration comparison ─────────────────────────────────────────
    # Pick pairs of different classes, ask which is longer
    pairs = [
        (a, b)
        for i, a in enumerate(events)
        for b in events[i+1:]
        if a["event"] != b["event"]
        and a["duration_sec"] != b["duration_sec"]
    ]
    random.shuffle(pairs)
    for a, b in pairs[:n_per_family]:
        longer = a["event_description"] if a["duration_sec"] > b["duration_sec"] else b["event_description"]
        questions.append({
            "family":    "duration_comparison",
            "question":  f"Which activity lasts longer: '{a['event_description']}' or '{b['event_description']}'?",
            "answer":    longer,
            "event_a":   a["event_description"],
            "event_b":   b["event_description"],
            "dur_a_sec": a["duration_sec"],
            "dur_b_sec": b["duration_sec"],
        })

    # ── Family 4: Composite activity identification ───────────────────────────
    # Find time windows where multiple events overlap
    # Sample a few such windows and ask what's happening
    for e in events[:n_per_family]:
        t_start = e["start_frame"] / fps
        t_end   = e["end_frame"]   / fps
        # Find all events overlapping this one
        co_occurring = [
            x["event_description"] for x in events
            if x["start_frame"] < e["end_frame"]
            and x["end_frame"]  > e["start_frame"]
        ]
        if len(co_occurring) >= 2:
            questions.append({
                "family":   "composite",
                "question": f"What activities are happening between "
                            f"{sec_to_mmss(t_start)} and {sec_to_mmss(t_end)}?",
                "answer":   sorted(set(co_occurring)),
                "t_start":  t_start,
                "t_end":    t_end,
            })

    return questions

# Generate for all clips
all_questions = {}
for vid_id, clip in clips.items():
    qs = generate_questions(clip, n_per_family=5)
    all_questions[vid_id] = qs
    family_counts = {}
    for q in qs:
        family_counts[q["family"]] = family_counts.get(q["family"], 0) + 1
    print(f"{vid_id}: {len(qs)} questions — {family_counts}")

In [ ]:
print("Loading model...")
t0 = time.time()

model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    attn_implementation="sdpa", 
    device_map="auto",
)
processor = AutoProcessor.from_pretrained(MODEL_ID)

print(f"Done in {time.time()-t0:.1f}s")
print(f"Device map: {model.hf_device_map}")

In [ ]:
def build_qa_prompt(question: str, 
                    frame_indices: np.ndarray, 
                    has_video: bool,
                    dataset_fps: int = DATASET_FPS) -> str:
    if not has_video:
        return (
            f"You are analyzing a home activity video.\n\n"
            f"Question: {question}\n\n"
            f"Answer concisely in one sentence."
        )

    frame_lines = "\n".join(
        f"  frame {i+1:02d} → index {int(f):6d} → "
        f"{int(f/dataset_fps)//60:02d}:{(f/dataset_fps)%60:05.2f}"
        for i, f in enumerate(frame_indices)
    )
    return (
        f"You are analyzing a home activity video recorded at {dataset_fps} fps.\n\n"
        f"The frames you see correspond to:\n{frame_lines}\n\n"
        f"Question: {question}\n\n"
        f"Answer concisely in one sentence."
    )

def run_qa(frames: list, 
           question: str,
           frame_indices: np.ndarray,
           model, processor,
           has_video: bool = True,
           max_new_tokens: int = 128) -> str:

    prompt = build_qa_prompt(question, frame_indices, has_video)

    if has_video and len(frames) > 0:
        messages = [{
            "role": "user",
            "content": [
                {"type": "video", "video": frames},
                {"type": "text",  "text": prompt},
            ]
        }]
    else:
        messages = [{
            "role": "user",
            "content": prompt,
        }]

    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    
    if has_video and len(frames) > 0:
        images, videos, video_kwargs = process_vision_info(
            messages, return_video_kwargs=True, return_video_metadata=True
        )
        
        if videos is not None:
            videos, video_metadatas = zip(*videos)
            videos, video_metadatas = list(videos), list(video_metadatas)
            # update metadata with global informations
            video_metadatas[0]["fps"] = DATASET_FPS
        else:
            video_metadatas = None
    
        if video_kwargs:
            video_kwargs = {
                k: v[0] if isinstance(v, list) and len(v) == 1 else v
                for k, v in video_kwargs.items()
            }
            
        inputs = processor(
            text=[text],
            images=images,
            videos=videos,
            video_metadata=video_metadatas,
            padding=True,
            return_tensors="pt",
            **video_kwargs,
        ).to(model.device)
        
    else:
        inputs = processor(
            text=[text], padding=True, return_tensors="pt"
        ).to(model.device)

    print(f"  Input tokens: {inputs['input_ids'].shape[-1]:,}")
    
    t0 = time.time()
    with torch.no_grad():
        output_ids = model.generate(
            **inputs, 
            max_new_tokens=max_new_tokens, 
            do_sample=False
        )
        
    
    generated = output_ids[:, inputs["input_ids"].shape[1]:]
    print(f"  Generated {generated.shape[-1]} words in {time.time()-t0:.1f}s")
    
    return processor.batch_decode(generated, skip_special_tokens=True)[0].strip()


def run_all_conditions(vid_id: str, questions: list, 
                       clip_frames: dict, model, processor) -> list:
    frames_data = clip_frames[vid_id]
    results = []

    for q in questions:
        print(f"  [{q['family']}] {q['question'][:60]}...")
        result = {"question": q, "responses": {}}

        for condition in ["normal", "shuffled", "blind"]:
            frames   = frames_data[condition]
            has_video = condition != "blind"
            response = run_qa(
                frames, q["question"], frames_data["indices"],
                model, processor, has_video=has_video
            )
            result["responses"][condition] = response
            print(f"    {condition:10s}: {response[:80]}")

        results.append(result)

    return results

# Run for all videos
all_results = {}
for vid_id in VIDEO_IDS:
    print(f"\n{'='*60}\n{vid_id}\n{'='*60}")
    all_results[vid_id] = run_all_conditions(
        vid_id, all_questions[vid_id], clip_frames, model, processor
    )

# Save everything
import json
with open(f"{OUTPUT_DIR}/qa_results.json", "w") as f:
    json.dump(all_results, f, indent=2, default=str)
print("\nSaved to qa_results.json")